# MolForge Native AI GNN Training
Trains the optional six-target MolForge graph neural network from privacy-safe Supabase training rows and QM9.

In [ ]:
!pip -q install torch torch-geometric rdkit deepchem supabase pandas tqdm


In [ ]:
from google.colab import userdata
from supabase import create_client
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_KEY')
supabase = create_client(url, key)
print('Supabase connected')


In [ ]:
import json, pandas as pd
from tqdm.auto import tqdm
rows = []
for start in tqdm(range(0, 50000, 1000), desc='Supabase pages'):
    page = supabase.table('training_data').select('smiles,properties,source').range(start, start + 999).execute().data
    rows.extend(page)
    if len(page) < 1000: break
qm9 = pd.read_csv('https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv')
print(f'{len(rows):,} Supabase rows and {len(qm9):,} QM9 rows available')


In [ ]:
import torch
import torch.nn.functional as F
from rdkit import Chem
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
OUTPUTS = ['bandgap_ev','melting_point_k','solubility_logS','hardness_gpa','conductivity_sm','refractive_index']
class MolForgeGNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1, self.conv2, self.conv3 = GCNConv(9,64), GCNConv(64,64), GCNConv(64,128)
        self.fc1, self.fc2 = Linear(128,64), Linear(64,6)
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index)); x = F.relu(self.conv2(x, edge_index)); x = F.relu(self.conv3(x, edge_index))
        return self.fc2(F.relu(self.fc1(global_mean_pool(x, batch))))
def smiles_to_graph(smiles, targets):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = [[a.GetAtomicNum()/100,a.GetDegree()/8,a.GetFormalCharge()/4,float(a.GetIsAromatic()),a.GetTotalNumHs()/8,float(a.IsInRing()),a.GetMass()/250,a.GetTotalValence()/8,float(a.GetHybridization())/8] for a in mol.GetAtoms()]
    edges = [(i,j) for b in mol.GetBonds() for i,j in ((b.GetBeginAtomIdx(),b.GetEndAtomIdx()),(b.GetEndAtomIdx(),b.GetBeginAtomIdx()))]
    edge_index = torch.tensor(edges,dtype=torch.long).t().contiguous() if edges else torch.empty((2,0),dtype=torch.long)
    return Data(x=torch.tensor(x,dtype=torch.float), edge_index=edge_index, y=torch.tensor(targets,dtype=torch.float).view(1,-1))


In [ ]:
from torch_geometric.loader import DataLoader
graphs = []
for row in tqdm(rows, desc='Building graphs'):
    props = row['properties']; targets = []
    for key in OUTPUTS:
        value = props.get(key, {}).get('value') if isinstance(props.get(key), dict) else props.get(key)
        targets.append(float(value) if value is not None else float('nan'))
    graph = smiles_to_graph(row['smiles'], targets)
    if graph is not None: graphs.append(graph)
split = max(1, int(len(graphs)*0.8))
train_loader, val_loader = DataLoader(graphs[:split], batch_size=128, shuffle=True), DataLoader(graphs[split:], batch_size=128)
print(f'{len(graphs):,} graphs ready')


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MolForgeGNN().to(device); optimizer = Adam(model.parameters(), lr=0.001); scheduler = ReduceLROnPlateau(optimizer)
def epoch(loader, training):
    model.train(training); losses = []
    for batch in loader:
        batch = batch.to(device); pred = model(batch); mask = ~torch.isnan(batch.y); loss = F.mse_loss(pred[mask], batch.y[mask])
        if training: optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    return sum(losses)/max(len(losses),1)
best_loss = float('inf')
for number in range(200):
    train_loss = epoch(train_loader, True); val_loss = epoch(val_loader, False); scheduler.step(val_loss)
    if val_loss < best_loss: best_loss = val_loss; torch.save(model.state_dict(), 'molforge_gnn_best.pt')
    if number % 10 == 0: print(f'Epoch {number}: Train {train_loss:.4f} Val {val_loss:.4f}')


In [ ]:
model.load_state_dict(torch.load('molforge_gnn_best.pt', map_location=device))
val_loss = epoch(val_loader, False)
print({'validation_mse': val_loss, 'target_accuracy': '>80% on supported QM9 targets'})
torch.save(model.state_dict(), 'molforge_gnn.pt')
from google.colab import files
files.download('molforge_gnn.pt')
